# recs_009 - Retrieval Pipeline vs Baseline Parity

Purpose: validate that latest retrieval pipeline outputs match the frozen baseline within tolerance for baseline methods.

## Prerequisites

1. Run retrieval eval job to generate latest artifacts:
   - `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json`
2. Ensure baseline exists (or create/update it):
   - `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json --write-baseline`
3. Confirm these files exist under `artifacts/recs/retrieval/runs/latest/`:
   - `eval_retrieval_overall.csv`
   - `eval_retrieval_baseline_overall.json`

This notebook compares:
- Current pipeline table: `artifacts/recs/retrieval/runs/latest/eval_retrieval_overall.csv`
- Frozen baseline JSON: `artifacts/recs/retrieval/runs/latest/eval_retrieval_baseline_overall.json`

Core checked metrics:
- `Hit@K`, `Recall@K`, `MAP@K`, `NDCG@K`, `MRR`

Core checked methods:
- `raw`, `popularity_train`, `multi_mean_train`

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


# --- config ---
REPO_ROOT = _find_repo_root(Path.cwd())
PIPELINE_OVERALL_PATH = REPO_ROOT / "artifacts" / "recs" / "retrieval" / "runs" / "latest" / "eval_retrieval_overall.csv"
BASELINE_JSON_PATH = REPO_ROOT / "artifacts" / "recs" / "retrieval" / "runs" / "latest" / "eval_retrieval_baseline_overall.json"

METHODS = ["raw", "popularity_train", "multi_mean_train"]
METRICS = ["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]
TOLERANCE = 1e-3

In [2]:
if not PIPELINE_OVERALL_PATH.is_file():
    raise FileNotFoundError(f"Missing pipeline overall CSV: {PIPELINE_OVERALL_PATH}")
if not BASELINE_JSON_PATH.is_file():
    raise FileNotFoundError(
        f"Missing frozen baseline JSON: {BASELINE_JSON_PATH}\n"
        "Create it with:\n"
        "python scripts/recs_job_eval_retrieval.py "
        "configs/recs_job_eval_retrieval.json --write-baseline"
    )

pipeline_df = pd.read_csv(PIPELINE_OVERALL_PATH)
payload = json.loads(BASELINE_JSON_PATH.read_text(encoding="utf-8"))
by_method = payload.get("overall_by_method", {})
if not by_method:
    raise ValueError(f"Baseline JSON missing overall_by_method: {BASELINE_JSON_PATH}")

rows = []
for method, metrics in by_method.items():
    row = {"method": method}
    for metric in METRICS:
        row[metric] = float(metrics[metric])
    rows.append(row)
reference_df = pd.DataFrame(rows)

print("Loaded:")
print("-", PIPELINE_OVERALL_PATH)
print("-", BASELINE_JSON_PATH)
print("pipeline rows:", len(pipeline_df), "baseline rows:", len(reference_df))

Loaded:
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_overall.csv
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_baseline_overall.json
pipeline rows: 3 baseline rows: 3


In [3]:
required_cols = ["method", *METRICS]

for c in required_cols:
    if c not in pipeline_df.columns:
        raise ValueError(f"Pipeline CSV missing required column: {c}")
    if c not in reference_df.columns:
        raise ValueError(f"Baseline reference missing required column: {c}")

pipeline_sub = pipeline_df[pipeline_df["method"].isin(METHODS)][required_cols].copy()
reference_sub = reference_df[reference_df["method"].isin(METHODS)][required_cols].copy()

missing_methods_pipeline = sorted(set(METHODS) - set(pipeline_sub["method"].astype(str)))
missing_methods_reference = sorted(set(METHODS) - set(reference_sub["method"].astype(str)))

if missing_methods_pipeline:
    raise ValueError(f"Pipeline CSV missing methods: {missing_methods_pipeline}")
if missing_methods_reference:
    raise ValueError(f"Baseline reference missing methods: {missing_methods_reference}")

In [4]:
compare_df = pipeline_sub.merge(
    reference_sub,
    on="method",
    suffixes=("_pipeline", "_reference"),
    validate="one_to_one",
)

for metric in METRICS:
    compare_df[f"{metric}_delta"] = compare_df[f"{metric}_pipeline"] - compare_df[f"{metric}_reference"]
    compare_df[f"{metric}_abs_delta"] = compare_df[f"{metric}_delta"].abs()

abs_delta_cols = [f"{m}_abs_delta" for m in METRICS]
compare_df["max_abs_delta"] = compare_df[abs_delta_cols].max(axis=1)

summary_cols = ["method", "max_abs_delta", *abs_delta_cols]
display(compare_df[summary_cols].sort_values("max_abs_delta", ascending=False))

,method,max_abs_delta,Hit@K_abs_delta,Recall@K_abs_delta,MAP@K_abs_delta,NDCG@K_abs_delta,MRR_abs_delta
2,raw,9.020562e-17,0.0,4.163336e-17,1.387779e-17,7.632783e-17,9.020562e-17
1,multi_mean_train,8.326673e-17,0.0,1.387779e-17,7.285839e-17,2.775558e-17,8.326673e-17
0,popularity_train,6.938894e-17,0.0,2.775558e-17,6.938894e-18,1.387779e-17,6.938894e-17


In [5]:
violations = []
for _, row in compare_df.iterrows():
    method = str(row["method"])
    for metric in METRICS:
        abs_delta = float(row[f"{metric}_abs_delta"])
        if abs_delta > TOLERANCE:
            violations.append((method, metric, abs_delta))

if violations:
    msg = "\n".join([f"{m}.{metric}: abs_delta={d:.6f} > tol={TOLERANCE:.6f}" for m, metric, d in violations])
    raise AssertionError("Parity check failed:\n" + msg)

print(f"PASS: pipeline/reference parity within tolerance={TOLERANCE:.6f}")

PASS: pipeline/reference parity within tolerance=0.001000
